In [1]:
from __future__ import annotations
!uv add paddleocr paddlepaddle
import re
import json
import os
import tempfile
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Protocol, Tuple

import faiss
import pymupdf
import gradio as gr
import numpy as np
from llama_cpp import Llama
from llama_index.core import Document, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from sentence_transformers import SentenceTransformer

class InstructLLM(Protocol):
    def complete(self, prompt: str, max_tokens: int = 512) -> str: ...


class OcrLLM(Protocol):
    def transcribe(self, image_path: str) -> str: ...


class OcrError(RuntimeError):
    """A page image could not be transcribed by the configured OCR model."""


class OcrModelUnavailable(OcrError):
    """The configured OCR weights could not be loaded."""


class LlamaCppInstruct:
    """Chat completion against one GGUF instruct model."""

    def __init__(
        self,
        repo_id: str,
        filename: str,
        n_ctx: int = 5096,
        n_gpu_layers: int = -1,
        temperature: float = 0.0,
    ):
        self.repo_id = repo_id
        self.filename = filename
        self.temperature = temperature
        self.llm = Llama.from_pretrained(
            repo_id=repo_id,
            filename=filename,
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            verbose=False,
        )

    def complete(self, prompt: str, max_tokens: int = 512) -> str:
        output = self.llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=self.temperature,
        )
        return output["choices"][0]["message"]["content"] or ""


class PaddlePageOcr:
    """Transcribe a page image with PaddleOCR. Constructing this does not load the model."""

    def __init__(self):
        self._engine = None
        self._load_error: OcrModelUnavailable | None = None

    def _ensure_loaded(self):
        if self._load_error is not None:
            raise self._load_error
        if self._engine is not None:
            return self._engine
        try:
            from paddleocr import PaddleOCR
            try:
                self._engine = PaddleOCR(lang="en")
            except TypeError:
                self._engine = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)
        except Exception as exc:
            self._load_error = OcrModelUnavailable(f"Could not load PaddleOCR: {exc}")
            raise self._load_error from exc
        print("PaddleOCR ready.")
        return self._engine

    def transcribe(self, image_path: str) -> str:
        engine = self._ensure_loaded()
        try:
            if hasattr(engine, "predict"):
                raw = engine.predict(image_path)
            else:
                raw = engine.ocr(image_path, cls=True)
        except TypeError:
            raw = engine.ocr(image_path)
        except Exception as exc:
            raise OcrError(f"PaddleOCR failed on {image_path}: {exc}") from exc
        return "\n".join(_paddle_lines(raw)).strip()


def _paddle_lines(raw) -> List[str]:
    lines: List[str] = []
    if not raw:
        return lines
    items = raw if isinstance(raw, list) else [raw]
    for item in items:
        texts = getattr(item, "rec_texts", None)
        if texts is None and isinstance(item, dict):
            texts = item.get("rec_texts")
        if texts:
            lines.extend(str(text) for text in texts if text)
            continue
        if not isinstance(item, list):
            continue
        for row in item:
            if not row or not isinstance(row, (list, tuple)) or len(row) < 2:
                continue
            text_part = row[1]
            if isinstance(text_part, (list, tuple)) and text_part:
                lines.append(str(text_part[0]))
            elif isinstance(text_part, str):
                lines.append(text_part)
    return lines


UNPARSED_LABEL = "Unlabeled"
OCR_RENDER_DPI = 150


@dataclass
class PageClassification:
    doc_type: str
    continues_previous: bool
    keep_page_intact: bool
    fields: Dict[str, str]


@dataclass
class PageInfo:
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0
    fields: Dict[str, str] = field(default_factory=dict)
    keep_page_intact: bool = False
    ocr_error: Optional[str] = None


@dataclass
class LogicalDocument:
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    pages: List[PageInfo] = field(default_factory=list)
    fields: Dict[str, str] = field(default_factory=dict)
    chunks: Optional[List["ChunkMetadata"]] = None


@dataclass
class ChunkMetadata:
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    fields: Dict[str, str] = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None


MIN_EMBEDDED_CHARS = 50
_ADDRESS_RE = re.compile(
    r"(@|https?://|www\.|\.(com|org|net)\b|"
    r"\b(tel|phone|fax|e-mail|email)\b|"
    r"\b(street|avenue|ave\.?|road|rd\.?|boulevard|blvd\.?|way|lane|drive|dr\.?)\b|"
    r"\b[A-Z]{2}\s+\d{5}(?:-\d{4})?\b|"
    r",\s*[A-Z]{2}\s+\d{5})",
    re.I,
)
_PAGE_LINE_RE = re.compile(r"^page\s+\d+\s*(/|of)\s*\d+$", re.I)
_DATE_RE = re.compile(
    r"^(\d{1,2}[\s/-][A-Za-z]{3,9}[\s,/-]+\d{2,4}|\d{4}-\d{2}-\d{2}|\d{1,2}/\d{1,2}/\d{2,4})$"
)
_COMPANY_RE = re.compile(r"\b(inc|llc|ltd|corp|gmbh|ab|plc)\.?\s*$", re.I)
_SKIP_EXACT = {"united states", "usa", "u.s.a."}
_SMALL_WORDS = {"of", "and", "the", "for", "regarding", "with", "in", "on", "a", "to"}
_ROUTE_STOP = {"the", "and", "for", "with", "from", "that", "this", "what", "when", "where", "which", "page", "date", "does", "have"}


def _is_letterhead(line: str) -> bool:
    stripped = line.strip()
    if not stripped or stripped.casefold() in _SKIP_EXACT:
        return True
    if set(stripped) <= set("-_=* "):
        return True
    if _DATE_RE.match(stripped) or _PAGE_LINE_RE.match(stripped):
        return True
    if _ADDRESS_RE.search(stripped) or _COMPANY_RE.search(stripped):
        return True
    if stripped.endswith(":") and len(stripped) <= 24:
        return True
    return False


def _headingish(line: str) -> bool:
    words = re.findall(r"[A-Za-z][A-Za-z'/-]*", line)
    if len(words) < 2:
        return False
    good = sum(1 for word in words if word.isupper() or word[:1].isupper() or word.casefold() in _SMALL_WORDS)
    return good / len(words) >= 0.7


def _title_candidate(line: str) -> bool:
    if "docusign" in line.casefold() or re.fullmatch(r"table of contents", line.strip(), re.I):
        return False
    if line[:1].isdigit() or _is_letterhead(line) or len(line) > 110:
        return False
    if line.endswith(".") and len(line.split()) > 6:
        return False
    if not any(ch.isalpha() for ch in line):
        return False
    return _headingish(line) or (":" in line and not line.endswith(":"))


def _normalize_type(title: str) -> str:
    title = re.sub(r"\(continued\)", "", title, flags=re.I)
    title = re.sub(r"\bNo\.\s*\S+", "", title, flags=re.I)
    title = re.sub(r",?\s*Page\s+\d+\s+of\s+\d+", "", title, flags=re.I)
    return " ".join(title.split())


def _header_fields(lines: List[str]) -> Dict[str, str]:
    fields: Dict[str, str] = {}
    i = 0
    while i < len(lines):
        line = lines[i]
        if ":" in line and not line.endswith(":"):
            key, value = line.split(":", 1)
            key, value = key.strip(), value.strip()
            if _keep_field(key, value):
                fields.setdefault(key, value)
            i += 1
            continue
        if line.endswith(":") and i + 1 < len(lines):
            key = line[:-1].strip()
            value = lines[i + 1].strip()
            if value and not value.endswith(":") and _keep_field(key, value):
                fields.setdefault(key, value)
                i += 2
                continue
        i += 1
    return fields


def _keep_field(key: str, value: str) -> bool:
    if len(key) < 4 or len(key) > 40 or not value or len(value) > 160:
        return False
    if key[0].isdigit() or not any(ch.isalpha() for ch in key):
        return False
    if "docusign" in key.casefold() or "docusign" in value.casefold():
        return False
    if _is_letterhead(key) or _is_letterhead(value) or _ADDRESS_RE.search(value):
        return False
    return True


def _without_docusign(text: str) -> str:
    return "\n".join(line for line in (text or "").splitlines() if "docusign" not in line.casefold())


def _toc_entries(text: str) -> List[Tuple[str, int]]:
    lines = [line.strip() for line in _without_docusign(text).splitlines() if line.strip()]
    if not any(re.search(r"table of contents", line, re.I) for line in lines):
        return []
    entries: List[Tuple[str, int]] = []
    index = 0
    while index < len(lines):
        line = lines[index]
        if re.search(r"table of contents", line, re.I):
            index += 1
            continue
        cleaned = re.sub(r"[.·•…]{2,}", " ", line)
        cleaned = " ".join(cleaned.replace(":", " ").split()).strip(" .")
        matched = re.match(r"^(\d{1,2})[.)]?\s+(.+?)\s+(\d{1,3})$", cleaned)
        if matched:
            title = " ".join(matched.group(2).split()).strip(" .")
            page_no = int(matched.group(3))
            if _toc_title_ok(title) and 1 <= page_no <= 200:
                entries.append((title, page_no))
            index += 1
            continue
        title_only = re.match(r"^(\d{1,2})[.)]?\s+(.+)$", cleaned)
        if title_only and index + 1 < len(lines) and re.fullmatch(r"\d{1,3}", lines[index + 1]):
            title = " ".join(title_only.group(2).split()).strip(" .")
            page_no = int(lines[index + 1])
            if _toc_title_ok(title) and 1 <= page_no <= 200:
                entries.append((title, page_no))
            index += 2
            continue
        index += 1
    return entries if len(entries) >= 2 else []


def _toc_title_ok(title: str) -> bool:
    if len(title) < 3 or len(title) > 80:
        return False
    folded = title.casefold()
    if "docusign" in folded or folded == "table of contents":
        return False
    return any(ch.isalpha() for ch in title)


def _printed_page(text: str) -> Optional[Tuple[int, int]]:
    matched = re.search(r"page\s+(\d+)\s+of\s+(\d+)", _without_docusign(text), re.I)
    if not matched:
        return None
    return int(matched.group(1)), int(matched.group(2))


def _section_label(parent: str, titles: List[str]) -> str:
    body = "; ".join(titles)
    parent = (parent or "").strip()
    if parent and parent != UNPARSED_LABEL and "table of contents" not in parent.casefold() and "docusign" not in parent.casefold():
        if parent.casefold() not in body.casefold():
            body = f"{parent}: {body}"
    return body[:120]


def apply_toc_labels(pages: List[PageInfo]) -> None:
    claimed = set()
    for index, page in enumerate(pages):
        if index in claimed:
            continue
        entries = _toc_entries(page.text)
        if not entries:
            continue
        printed = _printed_page(page.text)
        origin = printed[0] if printed else 1
        last_printed = printed[1] if printed else max(page_no for _, page_no in entries)
        if last_printed < origin:
            continue
        parent = page.doc_type or ""
        if "docusign" in parent.casefold() or re.fullmatch(r"table of contents", parent.strip(), re.I):
            parent = ""
        by_offset: Dict[int, List[str]] = {}
        for title, page_no in entries:
            offset = page_no - origin
            if offset <= 0 or offset > last_printed - origin:
                continue
            by_offset.setdefault(offset, []).append(title)
        open_titles: List[str] = []
        for offset in range(0, last_printed - origin + 1):
            target = index + offset
            if target >= len(pages):
                break
            if target in claimed and target != index:
                break
            claimed.add(target)
            pages[target].page_in_doc = offset
            if offset == 0:
                if parent:
                    pages[target].doc_type = parent[:120]
                continue
            if offset in by_offset:
                open_titles = by_offset[offset]
            if open_titles:
                pages[target].doc_type = _section_label(parent, open_titles)


def parse_header_metadata(text: str) -> PageClassification:
    text = _without_docusign(text)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    header = lines[:18]
    title = ""
    raw = ""
    index = 0
    while index < len(header):
        line = header[index]
        if not _title_candidate(line):
            index += 1
            continue
        nxt = header[index + 1] if index + 1 < len(header) else ""
        if nxt and (_ADDRESS_RE.search(nxt) or _COMPANY_RE.search(nxt)):
            index += 1
            continue
        parts = [line]
        if (
            len(line) >= 42
            and nxt
            and ":" not in nxt
            and _title_candidate(nxt)
            and not nxt.endswith(",")
            and "whom" not in nxt.casefold()
        ):
            parts.append(nxt)
        raw = " ".join(parts)
        title = _normalize_type(raw)[:120]
        break
    if not title:
        title = UNPARSED_LABEL
    return PageClassification(
        doc_type=title,
        continues_previous="continued" in raw.casefold(),
        keep_page_intact=False,
        fields=_header_fields(lines),
    )


def _clean_label(reply: str) -> str:
    if not reply or not reply.strip():
        return ""
    line = reply.strip().splitlines()[0].strip().strip("\"'`")
    for prefix in ("label:", "document:", "page:"):
        if line.casefold().startswith(prefix):
            line = line[len(prefix):].strip()
    return " ".join(line.split())[:120]


class HeaderMetadata:
    """The instruct model reads the whole page and chooses the label. DocuSign text is left out."""

    def __init__(self, instruct: InstructLLM):
        self.instruct = instruct

    def classify_page(self, text: str, previous_text: str = "", page_num: int = 0, total_pages: int = 1) -> PageClassification:
        found = parse_header_metadata(text)
        label = self._label_with_model(text, page_num, total_pages)
        if label:
            found.doc_type = label
        if page_num == 0:
            found.continues_previous = False
        found.keep_page_intact = len(found.fields) >= 3
        return found

    def _label_with_model(self, text: str, page_num: int, total_pages: int) -> str:
        body = _without_docusign(text).strip()
        if not body:
            return ""
        prompt = (
            "Read this entire page. Choose one short label for it.\n"
            "Use the page's own heading and what the page is about.\n"
            "Do not use DocuSign text, envelope IDs, or signature stamps.\n"
            "Return only the label.\n\n"
            f"Page {page_num + 1} of {total_pages}\n\n"
            f"{body}"
        )
        try:
            reply = self.instruct.complete(prompt, max_tokens=48)
        except Exception as exc:
            print(f"  Page {page_num + 1}: label error - {exc}")
            return ""
        label = _clean_label(reply)
        if not label or "docusign" in label.casefold():
            return ""
        return label

    def route(self, query: str, doc_types: List[str], fields_by_type: Optional[Dict[str, Dict[str, str]]] = None) -> Optional[str]:
        unique = list(dict.fromkeys(doc_type for doc_type in doc_types if doc_type))
        if not unique:
            return None
        if len(unique) == 1:
            return unique[0]
        query_words = {word for word in re.findall(r"[a-z0-9]{3,}", query.casefold()) if word not in _ROUTE_STOP}
        best_type = None
        best_score = 0
        for doc_type in unique:
            type_words = set(re.findall(r"[a-z0-9]{3,}", doc_type.casefold()))
            score = len(query_words & type_words)
            if score > best_score:
                best_type = doc_type
                best_score = score
        return best_type if best_score else None


def _pdf_path(pdf_file) -> str:
    return pdf_file.name if hasattr(pdf_file, "name") else pdf_file


def _render_page(page, page_idx: int, dpi: int) -> str:
    tmp_png = os.path.join(tempfile.gettempdir(), f"ocr_page_{page_idx}_{os.getpid()}.png")
    pixmap = page.get_pixmap(dpi=dpi)
    pixmap.save(tmp_png)
    return tmp_png


def extract_and_analyze_pdf(
    pdf_file,
    ocr: OcrLLM,
    classifier: HeaderMetadata,
    render_dpi: int = OCR_RENDER_DPI,
) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    print("Starting PDF extraction and analysis...")
    file_path = _pdf_path(pdf_file)
    fitz_doc = fitz.open(file_path)
    try:
        total_pages = fitz_doc.page_count
        pages_info: List[PageInfo] = []
        for i, page in enumerate(fitz_doc):
            text = (page.get_text("text") or "").strip()
            ocr_error = None
            if len(text) >= MIN_EMBEDDED_CHARS:
                print(f"  Page {i + 1}: embedded text {len(text)} characters.")
            else:
                print(f"  Page {i + 1}: {len(text)} embedded chars, running PaddleOCR...")
                tmp_png = _render_page(page, i, render_dpi)
                try:
                    text = ocr.transcribe(tmp_png).strip()
                    print(f"  Page {i + 1}: PaddleOCR extracted {len(text)} characters.")
                except OcrModelUnavailable:
                    raise
                except Exception as exc:
                    text = ""
                    ocr_error = str(exc)
                    print(f"  Page {i + 1}: PaddleOCR failed - {exc}")
                finally:
                    if os.path.exists(tmp_png):
                        try:
                            os.remove(tmp_png)
                        except OSError:
                            pass
            pages_info.append(PageInfo(page_num=i, text=text, ocr_error=ocr_error))
    finally:
        fitz_doc.close()

    if not pages_info:
        raise ValueError("No pages could be extracted from PDF")

    for i, page_obj in enumerate(pages_info):
        previous_text = pages_info[i - 1].text if i else ""
        classified = classifier.classify_page(
            page_obj.text,
            previous_text,
            page_num=i,
            total_pages=total_pages,
        )
        continued = (
            i > 0
            and classified.continues_previous
            and classified.doc_type == pages_info[i - 1].doc_type
            and classified.doc_type != UNPARSED_LABEL
        )
        if i == 0 or not continued:
            page_obj.page_in_doc = 0
            page_obj.doc_type = classified.doc_type
        else:
            page_obj.page_in_doc = pages_info[i - 1].page_in_doc + 1
            page_obj.doc_type = pages_info[i - 1].doc_type
        page_obj.fields = classified.fields
        page_obj.keep_page_intact = classified.keep_page_intact

    for i, page_obj in enumerate(pages_info):
        if "docusign" in (page_obj.doc_type or "").casefold():
            page_obj.doc_type = UNPARSED_LABEL
        page_obj.fields = {key: value for key, value in page_obj.fields.items() if "docusign" not in key.casefold() and "docusign" not in value.casefold()}
        print(f"  Page {i + 1}: {page_obj.doc_type} (page_in_doc={page_obj.page_in_doc})")

    logical_docs: List[LogicalDocument] = []
    current_pages: List[PageInfo] = []

    def flush() -> None:
        if not current_pages:
            return
        opening = current_pages[0]
        merged_fields: Dict[str, str] = {}
        for page in current_pages:
            for key, value in page.fields.items():
                merged_fields.setdefault(key, value)
        body = "\n\n".join(page_display_text(page) for page in current_pages if page.text.strip() or page.fields)
        logical_docs.append(
            LogicalDocument(
                doc_id=f"doc_{len(logical_docs)}",
                doc_type=opening.doc_type or UNPARSED_LABEL,
                page_start=opening.page_num,
                page_end=current_pages[-1].page_num,
                text=body.strip(),
                pages=list(current_pages),
                fields=merged_fields,
            )
        )

    for page in pages_info:
        if page.page_in_doc == 0 and current_pages:
            flush()
            current_pages = []
        current_pages.append(page)
    flush()
    return pages_info, logical_docs


def page_display_text(page: PageInfo) -> str:
    lines = [f"Document type: {page.doc_type}"]
    lines.extend(f"{key}: {value}" for key, value in page.fields.items())
    body = page.text.strip()
    if body:
        lines.extend(["", body])
    return "\n".join(lines).strip()


def _chunk_text(page: PageInfo, body: str) -> str:
    prefix = _field_prefix(page)
    body = (body or "").strip()
    if body:
        return f"{prefix}\n\n{body}"
    return prefix


def _append_chunk(
    chunks: List[ChunkMetadata],
    logical_doc: LogicalDocument,
    page: PageInfo,
    body: str,
) -> None:
    chunks.append(
        ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{len(chunks)}",
            doc_id=logical_doc.doc_id,
            doc_type=page.doc_type or logical_doc.doc_type,
            chunk_index=len(chunks),
            page_start=page.page_num,
            page_end=page.page_num,
            text=_chunk_text(page, body),
            fields=dict(page.fields),
        )
    )


def _field_prefix(page: PageInfo) -> str:
    lines = [f"Document type: {page.doc_type}"]
    lines.extend(f"{key}: {value}" for key, value in page.fields.items())
    return "\n".join(lines)


def chunk_document_with_metadata(
    logical_doc: LogicalDocument,
    chunk_size: int = 500,
    overlap: int = 100,
) -> List[ChunkMetadata]:
    chunks: List[ChunkMetadata] = []
    pages = logical_doc.pages or []
    if not pages:
        if logical_doc.text.strip():
            chunks.append(
                ChunkMetadata(
                    chunk_id=f"{logical_doc.doc_id}_chunk_0",
                    doc_id=logical_doc.doc_id,
                    doc_type=logical_doc.doc_type,
                    chunk_index=0,
                    page_start=logical_doc.page_start,
                    page_end=logical_doc.page_end,
                    text=logical_doc.text,
                    fields=dict(logical_doc.fields),
                )
            )
        return chunks

    for page in pages:
        body = page.text.strip()
        if not body and not page.fields:
            continue
        words = body.split()
        if page.keep_page_intact or len(words) <= chunk_size:
            _append_chunk(chunks, logical_doc, page, body)
            continue

        lines = body.splitlines()
        current_lines: List[str] = []
        current_words = 0
        for line in lines:
            line_word_count = len(line.split())
            if current_words + line_word_count > chunk_size and current_lines:
                _append_chunk(chunks, logical_doc, page, "\n".join(current_lines))
                overlap_lines: List[str] = []
                overlap_words = 0
                for prev_line in reversed(current_lines):
                    word_len = len(prev_line.split())
                    if overlap_words + word_len > overlap:
                        break
                    overlap_lines.insert(0, prev_line)
                    overlap_words += word_len
                current_lines = overlap_lines
                current_words = overlap_words
            current_lines.append(line)
            current_words += line_word_count
        if current_lines:
            _append_chunk(chunks, logical_doc, page, "\n".join(current_lines))
    return chunks


def process_all_documents(logical_docs: List[LogicalDocument]) -> List[ChunkMetadata]:
    all_chunks: List[ChunkMetadata] = []
    for logical_doc in logical_docs:
        chunks = chunk_document_with_metadata(logical_doc)
        logical_doc.chunks = chunks
        all_chunks.extend(chunks)
    return all_chunks


class IntelligentRetriever:
    def __init__(self, classifier: HeaderMetadata, embed_model, llama_embed_model):
        self.classifier = classifier
        self.embed_model = embed_model
        self.llama_embed_model = llama_embed_model
        self.index = None
        self.llama_index = None
        self.chunks_metadata: List[ChunkMetadata] = []
        self.doc_type_indices: Dict[str, dict] = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata], logical_docs: List[LogicalDocument]):
        self.chunks_metadata = chunks_metadata
        self.doc_type_indices = {}
        if not chunks_metadata:
            raise ValueError("OCR produced no indexable text")

        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = self.embed_model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
        embeddings = np.array(embeddings, dtype="float32")
        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        for doc_type in dict.fromkeys(chunk.doc_type for chunk in chunks_metadata):
            type_indices = [i for i, chunk in enumerate(chunks_metadata) if chunk.doc_type == doc_type]
            type_index = faiss.IndexFlatIP(dim)
            type_index.add(embeddings[type_indices])
            self.doc_type_indices[doc_type] = {"index": type_index, "mapping": type_indices}

        chunked_documents = [
            Document(
                text=doc.text,
                metadata={
                    "doc_id": doc.doc_id,
                    "doc_type": doc.doc_type,
                    "page_start": doc.page_start,
                    "page_end": doc.page_end,
                    "fields": json.dumps(doc.fields, ensure_ascii=False),
                },
            )
            for doc in logical_docs
            if doc.text.strip()
        ]
        if chunked_documents:
            self.llama_index = VectorStoreIndex.from_documents(
                chunked_documents,
                embed_model=self.llama_embed_model,
            )

    def retrieve(
        self,
        query: str,
        k: int = 4,
        filter_doc_type: Optional[str] = None,
        auto_route: bool = True,
    ) -> Tuple[List[Tuple[ChunkMetadata, float]], str]:
        if filter_doc_type and filter_doc_type != "All":
            target_type = filter_doc_type
            routed_label = f"Filter: {target_type}"
        elif auto_route:
            field_map: Dict[str, Dict[str, str]] = {}
            for chunk in self.chunks_metadata:
                field_map.setdefault(chunk.doc_type, {}).update(chunk.fields)
            target_type = self.classifier.route(query, list(self.doc_type_indices), field_map)
            routed_label = f"Auto-Routed: {target_type}" if target_type else "All Documents"
        else:
            target_type = None
            routed_label = "All Documents"

        query_embedding = np.array(
            self.embed_model.encode([query], normalize_embeddings=True),
            dtype="float32",
        )
        if target_type and target_type in self.doc_type_indices:
            type_data = self.doc_type_indices[target_type]
            search_k = min(k, len(type_data["mapping"]))
            distances, indices = type_data["index"].search(query_embedding, search_k)
            pairs = [
                (self.chunks_metadata[type_data["mapping"][i]], float(max(0.0, distances[0][idx])))
                for idx, i in enumerate(indices[0])
                if i != -1
            ]
            pairs.sort(key=lambda item: (item[0].page_start, item[0].chunk_index))
            return pairs, routed_label

        search_k = min(k, len(self.chunks_metadata))
        distances, indices = self.index.search(query_embedding, search_k)
        chunk_indices = [i for i in indices[0] if i != -1]
        scores = [float(max(0.0, score)) for score in distances[0][: len(chunk_indices)]]
        return (
            [(self.chunks_metadata[i], scores[idx]) for idx, i in enumerate(chunk_indices)],
            routed_label,
        )

    def retrieve_all_chunks(self, query: str) -> List[Tuple[ChunkMetadata, float]]:
        if not self.chunks_metadata or self.index is None:
            return []
        query_embedding = np.array(
            self.embed_model.encode([query], normalize_embeddings=True),
            dtype="float32",
        )
        distances, indices = self.index.search(query_embedding, len(self.chunks_metadata))
        chunk_indices = [i for i in indices[0] if i != -1]
        scores = [float(max(0.0, score)) for score in distances[0][: len(chunk_indices)]]
        return [(self.chunks_metadata[i], scores[idx]) for idx, i in enumerate(chunk_indices)]


class EnhancedDocumentStore:
    def __init__(self, ocr: OcrLLM, classifier: HeaderMetadata, embed_model, llama_embed_model, instruct: InstructLLM):
        self.ocr = ocr
        self.classifier = classifier
        self.pages_info: List[PageInfo] = []
        self.logical_docs: List[LogicalDocument] = []
        self.chunks_metadata: List[ChunkMetadata] = []
        self.instruct = instruct
        self.retriever = IntelligentRetriever(classifier, embed_model, llama_embed_model)
        self.is_ready = False
        self.processing_stats: dict = {}
        self.last_results_dict: dict = {}

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        self.is_ready = False
        start_time = datetime.now()
        try:
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(
                pdf_file, self.ocr, self.classifier
            )
            self.chunks_metadata = process_all_documents(self.logical_docs)
            self.retriever.build_indices(self.chunks_metadata, self.logical_docs)
            process_time = (datetime.now() - start_time).total_seconds()
            unique_types = list(dict.fromkeys(doc.doc_type for doc in self.logical_docs))
            ocr_failures = sum(1 for page in self.pages_info if page.ocr_error)
            self.processing_stats = {
                "filename": filename,
                "total_pages": len(self.pages_info),
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunks_metadata),
                "document_types": unique_types,
                "ocr_failures": ocr_failures,
                "processing_time": f"{process_time:.1f}s",
            }
            self.is_ready = True
            return True, self.processing_stats
        except Exception as exc:
            return False, {"error": str(exc)}

    @staticmethod
    def _is_answer_not_found(answer: str) -> bool:
        if not answer or not answer.strip() or answer.strip().lower().startswith("not_found"):
            return True
        not_found_phrases = [
            "not mentioned",
            "not provided",
            "does not contain",
            "no information",
            "cannot find",
            "unable to find",
        ]
        lowered = answer.strip().lower()
        return any(phrase in lowered for phrase in not_found_phrases)

    @staticmethod
    def _clean_not_found_prefix(answer: str) -> str:
        cleaned = answer.strip()
        if cleaned.upper().startswith("NOT_FOUND:"):
            return cleaned[10:].strip()
        if cleaned.upper().startswith("NOT_FOUND"):
            return cleaned[9:].strip(" :-")
        return cleaned

    def _generate_answer_from_chunks(self, question: str, chunks_with_scores: List[Tuple[ChunkMetadata, float]]) -> str:
        retrieved_text = "\n\n---\n\n".join(
            f"[Document: {chunk.doc_type} | Page {chunk.page_start + 1}]\n{chunk.text}"
            for chunk, _ in chunks_with_scores
        )
        prompt = (
            "Based ONLY on the following text, answer the user's question accurately. "
            "Copy dates, lot numbers, and other identifiers exactly as they appear. "
            "If the answer is not in the text, begin the response with \"NOT_FOUND:\".\n\n"
            f"Text:\n{retrieved_text}\n\nUser Question: {question}"
        )
        return self.instruct.complete(prompt, max_tokens=512).strip()

    def query(self, question: str, filter_type: Optional[str] = None, auto_route: bool = True, k: int = 4) -> Dict:
        if not self.is_ready:
            return {"answer": "⚠️ Please upload a PDF file first.", "sources": [], "filter_used": "none"}

        retrieved, routed_label = self.retriever.retrieve(
            question, k=k, filter_doc_type=filter_type, auto_route=auto_route
        )
        answer = self._generate_answer_from_chunks(question, retrieved) if retrieved else ""

        if not retrieved or self._is_answer_not_found(answer):
            for doc in self.logical_docs:
                if not doc.chunks:
                    continue
                doc_batch = [(chunk, 1.0) for chunk in doc.chunks]
                candidate_answer = self._generate_answer_from_chunks(question, doc_batch)
                if not self._is_answer_not_found(candidate_answer):
                    answer = self._clean_not_found_prefix(candidate_answer)
                    retrieved = doc_batch
                    routed_label += f" -> searched {doc.doc_type}"
                    break

        answer = self._clean_not_found_prefix(answer)
        unique_sources = list(
            dict.fromkeys(f"{chunk.doc_type} (Page {chunk.page_start + 1})" for chunk, _ in retrieved)
        )
        self.last_results_dict = {
            "query": question,
            "predicted_doc_type": routed_label,
            "matched_chunks": [
                {
                    "text": chunk.text,
                    "score": round(score, 4),
                    "metadata": {
                        "doc_id": chunk.doc_id,
                        "doc_type": chunk.doc_type,
                        "page_start": chunk.page_start,
                        "fields": chunk.fields,
                    },
                }
                for chunk, score in retrieved
            ],
            "answer": answer,
        }
        return {"answer": answer, "sources": unique_sources, "filter_used": routed_label}

# Instruct model answers questions. Page metadata does not call it.
instruct_llm = LlamaCppInstruct(
    repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF",
    filename="Mistral-7B-Instruct-v0.3-Q5_K_S.gguf",
)

# PaddleOCR runs only for a page with little or no embedded text.
ocr_llm = PaddlePageOcr()
classifier = HeaderMetadata(instruct_llm)
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
llama_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("Imports and configuration complete.")


def new_store():
    return EnhancedDocumentStore(ocr_llm, classifier, embed_model, llama_embed_model, instruct_llm)


doc_store = new_store()


def process_pdf_ui(file):
    global doc_store
    if file is None:
        doc_store = new_store()
        return "Waiting for PDF...", gr.update(choices=["All"], value="All"), "No document processed yet."
    success, stats = doc_store.process_pdf(file.name if hasattr(file, "name") else file)
    if not success:
        return f"❌ Error: {stats.get('error', 'Unknown')}", gr.update(choices=["All"], value="All"), "Error processing."
    failure_note = ""
    if stats.get("ocr_failures"):
        failure_note = f" {stats['ocr_failures']} page(s) had no OCR text."
    status_text = (
        f"✅ Processed {stats['total_pages']} pages into {stats['documents_found']} docs "
        f"({stats['processing_time']}).{failure_note}"
    )
    lines = []
    for i, doc in enumerate(doc_store.logical_docs):
        field_text = ""
        if doc.fields:
            shown = list(doc.fields.items())[:6]
            field_text = " — " + "; ".join(f"{key}: {value}" for key, value in shown)
        lines.append(
            f"{i + 1}. **{doc.doc_type}** — Pages {doc.page_start + 1} to {doc.page_end + 1}{field_text}"
        )
    breakdown_md = "\n".join(lines) if lines else "No document text extracted."
    return status_text, gr.update(choices=["All"] + stats["document_types"], value="All"), breakdown_md


def handle_chat(message, history, doc_filter, auto_route, top_k):
    history = history or []
    if not message or not message.strip():
        return history, ""
    if not doc_store.is_ready:
        return history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Please upload a PDF first."},
        ], ""
    res = doc_store.query(
        message,
        filter_type=None if doc_filter == "All" else doc_filter,
        auto_route=auto_route,
        k=int(top_k),
    )
    ans = res["answer"]
    if res.get("sources"):
        ans += f"\n\n*Sources: {', '.join(res['sources'])} ({res['filter_used']})*"
    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ans},
    ], ""


with gr.Blocks(title="Pharmaceutical Document Q&A") as demo:
    gr.Markdown("### 📄 Pharmaceutical Document Q&A")
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=380)
            user_query = gr.Textbox(placeholder="Ask a question...", label="Your Question")
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
            status_box = gr.Textbox(label="Status", interactive=False, value="Waiting for PDF...")
            with gr.Accordion("⚙️ Options", open=False):
                doc_filter = gr.Dropdown(choices=["All"], value="All", label="Filter")
                auto_route = gr.Checkbox(value=True, label="Auto-Route")
                top_k = gr.Dropdown(choices=[2, 4, 6, 8], value=4, label="Top K")
                detected_docs_md = gr.Markdown("No document processed yet.")
            with gr.Accordion("📥 Export", open=False):
                download_dict_btn = gr.Button("📥 Download JSON")
                download_chat_btn = gr.Button("💬 Download Chat")
                download_file_output = gr.File(label="Download File", interactive=False)

    pdf_input.change(fn=process_pdf_ui, inputs=pdf_input, outputs=[status_box, doc_filter, detected_docs_md])
    send_btn.click(fn=handle_chat, inputs=[user_query, chatbot, doc_filter, auto_route, top_k], outputs=[chatbot, user_query])
    user_query.submit(fn=handle_chat, inputs=[user_query, chatbot, doc_filter, auto_route, top_k], outputs=[chatbot, user_query])
    clear_btn.click(lambda: [], outputs=chatbot)
    download_dict_btn.click(fn=lambda: os.path.join(tempfile.gettempdir(), "rag_results.json"), outputs=download_file_output)
    download_chat_btn.click(fn=lambda history: os.path.join(tempfile.gettempdir(), "chat_history.json"), inputs=chatbot, outputs=download_file_output)

demo.launch()


Resolved 265 packages in 10ms
Checked 243 packages in 20ms


/Users/leo/Pfizer-RAG-AI-Pipeline/.venv/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
/Users/leo/Pfizer-RAG-AI-Pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 21:47:27.626000 8484 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/leo/Pfizer-RAG-AI-Pipeline/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|█████████████████████████████████████████████████████████████

Imports and configuration complete.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
